In [1]:
import pandas as pd
import numpy as np
import json
import re
from difflib import SequenceMatcher

file_path = "Abilities to Work Activities.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully!
Shape: (381, 4)

Columns:
['Abilities Element ID', 'Abilities Element Name', 'Work Activities Element ID', 'Work Activities Element Name']

First 5 rows:


,Abilities Element ID,Abilities Element Name,Work Activities Element ID,Work Activities Element Name
0,1.A.1.a.1,Oral Comprehension,4.A.1.a.1,Getting Information
1,1.A.1.a.1,Oral Comprehension,4.A.1.a.2,"Monitor Processes, Materials, or Surroundings"
2,1.A.1.a.1,Oral Comprehension,4.A.1.b.1,"Identifying Objects, Actions, and Events"
3,1.A.1.a.1,Oral Comprehension,4.A.2.a.1,"Judging the Qualities of Things, Services, or ..."
4,1.A.1.a.1,Oral Comprehension,4.A.2.a.2,Processing Information


In [2]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Missing values:
Abilities Element ID            0
Abilities Element Name          0
Work Activities Element ID      0
Work Activities Element Name    0
dtype: int64

Duplicate rows: 0


In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Cleaned columns:")
print(df.columns.tolist())

Cleaned columns:
['abilities_element_id', 'abilities_element_name', 'work_activities_element_id', 'work_activities_element_name']


In [4]:

df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (381, 4)


In [5]:
text_columns = [
    "abilities_element_name",
    "work_activities_element_name"
]

for column in text_columns:
    df[column] = (
        df[column]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Text cleaning completed.")

Text cleaning completed.


In [6]:
# 6. EXTRACT CANONICAL SKILLS


canonical_skills = sorted(
    df["abilities_element_name"]
    .dropna()
    .unique()
    .tolist()
)

print("Number of unique ability/skill categories:",
      len(canonical_skills))

print("\nFirst 20 canonical skills:")
print(canonical_skills[:20])

Number of unique ability/skill categories: 50

First 20 canonical skills:
['Arm-Hand Steadiness', 'Auditory Attention', 'Category Flexibility', 'Control Precision', 'Deductive Reasoning', 'Depth Perception', 'Dynamic Strength', 'Explosive Strength', 'Extent Flexibility', 'Far Vision', 'Finger Dexterity', 'Flexibility of Closure', 'Fluency of Ideas', 'Glare Sensitivity', 'Gross Body Coordination', 'Gross Body Equilibrium', 'Hearing Sensitivity', 'Inductive Reasoning', 'Information Ordering', 'Manual Dexterity']


In [7]:
# NORMALIZATION FUNCTION


def normalize_skill_text(skill):
    """
    Normalizes a skill before matching.
    """

    skill = str(skill).strip().lower()

    # Replace common separators
    skill = skill.replace("_", " ")
    skill = skill.replace("-", " ")

    # Remove extra spaces
    skill = re.sub(r"\s+", " ", skill)

    return skill

In [8]:

canonical_skill_map = {}

for skill in canonical_skills:
    normalized = normalize_skill_text(skill)
    canonical_skill_map[normalized] = skill

print("Canonical skill map created.")
print("Number of canonical entries:",
      len(canonical_skill_map))

Canonical skill map created.
Number of canonical entries: 50


In [9]:
#  CANONICAL ALIAS HASH MAP

# Project-documented aliases
alias_map = {

    "js": "JavaScript",
    "javascript": "JavaScript",
    "javascript es6": "JavaScript",
    "es6": "JavaScript",
    "node js": "JavaScript",

    "py": "Python",
    "python3": "Python",
    "python 3": "Python",

    "k8s": "Kubernetes",
    "kube": "Kubernetes",

    "scikit learn": "Scikit-Learn",
    "scikit-learn": "Scikit-Learn",
    "scikitleran": "Scikit-Learn",
    "scikit leran": "Scikit-Learn"
}


# Normalize alias keys
alias_map = {
    normalize_skill_text(alias): canonical
    for alias, canonical in alias_map.items()
}

print("Alias map created.")
print("Number of aliases:", len(alias_map))

Alias map created.
Number of aliases: 13


In [10]:
# 10. ADD DATASET SKILLS TO ALIAS LOOKUP


for normalized_skill, canonical_skill in canonical_skill_map.items():

    if normalized_skill not in alias_map:
        alias_map[normalized_skill] = canonical_skill

print("Final alias lookup size:", len(alias_map))


Final alias lookup size: 63


In [11]:
 # FUZZY MATCHING FUNCTION

FUZZY_THRESHOLD = 0.82


def fuzzy_similarity(text1, text2):
    """
    Calculates Gestalt similarity using SequenceMatcher.

    Returns a value between 0 and 1.
    """

    return SequenceMatcher(
        None,
        normalize_skill_text(text1),
        normalize_skill_text(text2)
    ).ratio()


In [12]:

def match_skill(skill):
    """
    Matches an input skill using:

    Step 1 -> Exact canonical/alias hash lookup
    Step 2 -> Fuzzy Gestalt matching
    Step 3 -> Return unmatched if similarity < 0.82
    """

    original_skill = str(skill).strip()

    if not original_skill:
        return {
            "input": original_skill,
            "canonical_skill": None,
            "method": "invalid",
            "similarity": 0.0
        }

    normalized_skill = normalize_skill_text(original_skill)

    # --------------------------------------------------------
    # Algorithm 1: O(1) Alias Hash Map Lookup
    # --------------------------------------------------------

    if normalized_skill in alias_map:

        canonical = alias_map[normalized_skill]

        return {
            "input": original_skill,
            "canonical_skill": canonical,
            "method": "alias/exact",
            "similarity": 1.0
        }

    # --------------------------------------------------------
    # Algorithm 2: Gestalt Fuzzy Matching
    # --------------------------------------------------------

    best_match = None
    best_score = 0.0

    for canonical_skill in canonical_skills:

        score = fuzzy_similarity(
            original_skill,
            canonical_skill
        )

        if score > best_score:
            best_score = score
            best_match = canonical_skill

    # --------------------------------------------------------
    # Apply documented threshold
    # --------------------------------------------------------

    if best_score >= FUZZY_THRESHOLD:

        return {
            "input": original_skill,
            "canonical_skill": best_match,
            "method": "fuzzy",
            "similarity": round(best_score, 4)
        }

    return {
        "input": original_skill,
        "canonical_skill": None,
        "method": "unmatched",
        "similarity": round(best_score, 4)
    }


In [13]:
# 13. TEST EXACT ALIAS MATCHING
# ============================================================

test_aliases = [
    "js",
    "javascript es6",
    "py",
    "python3",
    "k8s",
    "kube"
]

print("\nAlias Matching Tests")
print("=" * 60)

for skill in test_aliases:

    result = match_skill(skill)

    print(
        f"{result['input']:20} -> "
        f"{result['canonical_skill']} "
        f"({result['method']})"
    )




Alias Matching Tests
js                   -> JavaScript (alias/exact)
javascript es6       -> JavaScript (alias/exact)
py                   -> Python (alias/exact)
python3              -> Python (alias/exact)
k8s                  -> Kubernetes (alias/exact)
kube                 -> Kubernetes (alias/exact)


In [14]:
fuzzy_tests = [
    "scikit-leran",
    "pythn",
    "javscript"
]

print("\nFuzzy Matching Tests")
print("=" * 60)

for skill in fuzzy_tests:

    result = match_skill(skill)

    print(
        f"{result['input']:20} -> "
        f"{result['canonical_skill']} | "
        f"Method: {result['method']} | "
        f"Similarity: {result['similarity']}"
    )


Fuzzy Matching Tests
scikit-leran         -> Scikit-Learn | Method: alias/exact | Similarity: 1.0
pythn                -> None | Method: unmatched | Similarity: 0.381
javscript            -> None | Method: unmatched | Similarity: 0.4211


In [15]:

def match_skills(skills):
    """
    Matches a list of skills and removes duplicate
    canonical results.
    """

    results = []

    for skill in skills:

        result = match_skill(skill)

        if result["canonical_skill"] is not None:
            results.append(result)

    return results



In [17]:
# TEST MULTIPLE SKILLS
# ============================================================

student_skills = [
    "Python",
    "py",
    "python3",
    "js",
    "k8s",
    "scikit-leran"
]

results = match_skills(student_skills)

print("\nMultiple Skill Matching")
print("=" * 60)

for result in results:

    print(
        f"{result['input']:20} -> "
        f"{result['canonical_skill']:30} | "
        f"{result['method']:15} | "
        f"{result['similarity']}"
    )




Multiple Skill Matching
py                   -> Python                         | alias/exact     | 1.0
python3              -> Python                         | alias/exact     | 1.0
js                   -> JavaScript                     | alias/exact     | 1.0
k8s                  -> Kubernetes                     | alias/exact     | 1.0
scikit-leran         -> Scikit-Learn                   | alias/exact     | 1.0


In [18]:

matched_canonical_skills = sorted(
    set(
        result["canonical_skill"]
        for result in results
        if result["canonical_skill"] is not None
    )
)

print("\nUnique canonical skills:")
print(matched_canonical_skills)


Unique canonical skills:
['JavaScript', 'Kubernetes', 'Python', 'Scikit-Learn']


In [19]:
skill_work_activity_map = {}

for skill, group in df.groupby("abilities_element_name"):

    activities = sorted(
        group["work_activities_element_name"]
        .dropna()
        .unique()
        .tolist()
    )

    skill_work_activity_map[skill] = activities


print("\nExample Skill -> Work Activities mapping:")

for skill in list(skill_work_activity_map.keys())[:5]:

    print("\nSkill:", skill)

    for activity in skill_work_activity_map[skill][:5]:
        print("  -", activity)




Example Skill -> Work Activities mapping:

Skill: Arm-Hand Steadiness
  - Controlling Machines and Processes
  - Handling and Moving Objects
  - Operating Vehicles, Mechanized Devices, or Equipment
  - Repairing and Maintaining Electronic Equipment
  - Repairing and Maintaining Mechanical Equipment

Skill: Auditory Attention
  - Inspecting Equipment, Structures, or Material
  - Monitor Processes, Materials, or Surroundings
  - Operating Vehicles, Mechanized Devices, or Equipment
  - Performing for or Working Directly with the Public

Skill: Category Flexibility
  - Analyzing Data or Information
  - Coordinating the Work and Activities of Others
  - Developing Objectives and Strategies
  - Documenting/Recording Information
  - Identifying Objects, Actions, and Events

Skill: Control Precision
  - Controlling Machines and Processes
  - Operating Vehicles, Mechanized Devices, or Equipment
  - Repairing and Maintaining Electronic Equipment
  - Repairing and Maintaining Mechanical Equipmen

In [20]:
skills_data = []

for skill in canonical_skills:

    normalized_skill = normalize_skill_text(skill)

    skills_data.append({
        "skill": skill,
        "normalized_skill": normalized_skill,
        "work_activities": skill_work_activity_map.get(
            skill,
            []
        )
    })


print("\nNumber of taxonomy records:",
      len(skills_data))




Number of taxonomy records: 50


In [21]:
print("\nSample taxonomy records:")

for item in skills_data[:5]:

    print(json.dumps(
        item,
        indent=4,
        ensure_ascii=False
    ))


Sample taxonomy records:
{
    "skill": "Arm-Hand Steadiness",
    "normalized_skill": "arm hand steadiness",
    "work_activities": [
        "Controlling Machines and Processes",
        "Handling and Moving Objects",
        "Operating Vehicles, Mechanized Devices, or Equipment",
        "Repairing and Maintaining Electronic Equipment",
        "Repairing and Maintaining Mechanical Equipment"
    ]
}
{
    "skill": "Auditory Attention",
    "normalized_skill": "auditory attention",
    "work_activities": [
        "Inspecting Equipment, Structures, or Material",
        "Monitor Processes, Materials, or Surroundings",
        "Operating Vehicles, Mechanized Devices, or Equipment",
        "Performing for or Working Directly with the Public"
    ]
}
{
    "skill": "Category Flexibility",
    "normalized_skill": "category flexibility",
    "work_activities": [
        "Analyzing Data or Information",
        "Coordinating the Work and Activities of Others",
        "Developing Object

In [24]:
from pathlib import Path

output_dir = Path("processed")

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("\nOutput directory ready:", output_dir)


Output directory ready: processed


In [25]:
skills_df = pd.DataFrame(skills_data)

skills_csv_path = output_dir / "skills_processed.csv"

skills_df.to_csv(
    skills_csv_path,
    index=False,
    encoding="utf-8"
)

print(
    "Processed CSV saved:",
    skills_csv_path
)

Processed CSV saved: processed\skills_processed.csv


In [26]:
skills_json_path = output_dir / "skills.json"

with open(
    skills_json_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        skills_data,
        file,
        indent=4,
        ensure_ascii=False
    )

print(
    "skills.json created successfully!"
)

print(
    "Number of records:",
    len(skills_data)
)



skills.json created successfully!
Number of records: 50


In [27]:
with open(
    skills_json_path,
    "r",
    encoding="utf-8"
) as file:

    loaded_skills = json.load(file)

print("\nJSON verification successful!")

print(
    "Records loaded:",
    len(loaded_skills)
)



JSON verification successful!
Records loaded: 50


In [28]:
print("\nFirst JSON record:")

print(
    json.dumps(
        loaded_skills[0],
        indent=4,
        ensure_ascii=False
    )
)



First JSON record:
{
    "skill": "Arm-Hand Steadiness",
    "normalized_skill": "arm hand steadiness",
    "work_activities": [
        "Controlling Machines and Processes",
        "Handling and Moving Objects",
        "Operating Vehicles, Mechanized Devices, or Equipment",
        "Repairing and Maintaining Electronic Equipment",
        "Repairing and Maintaining Mechanical Equipment"
    ]
}


In [29]:
print("\n" + "=" * 60)
print("FINAL SKILL TAXONOMY VERIFICATION")
print("=" * 60)

print("Raw dataset rows:", len(df))
print("Canonical skill categories:", len(canonical_skills))
print("Alias mappings:", len(alias_map))
print("Fuzzy threshold:", FUZZY_THRESHOLD)
print("JSON records:", len(loaded_skills))
print("CSV file:", skills_csv_path)
print("JSON file:", skills_json_path)



FINAL SKILL TAXONOMY VERIFICATION
Raw dataset rows: 381
Canonical skill categories: 50
Alias mappings: 63
Fuzzy threshold: 0.82
JSON records: 50
CSV file: processed\skills_processed.csv
JSON file: processed\skills.json


In [30]:
def test_skill_matching():

    test_cases = [
        "js",
        "javascript es6",
        "py",
        "python3",
        "k8s",
        "kube",
        "scikit-leran"
    ]

    print("\nFINAL MATCHING TEST")
    print("=" * 70)

    for skill in test_cases:

        result = match_skill(skill)

        print(
            f"Input: {result['input']:<20} | "
            f"Canonical: {str(result['canonical_skill']):<25} | "
            f"Method: {result['method']:<15} | "
            f"Similarity: {result['similarity']}"
        )


test_skill_matching()


FINAL MATCHING TEST
Input: js                   | Canonical: JavaScript                | Method: alias/exact     | Similarity: 1.0
Input: javascript es6       | Canonical: JavaScript                | Method: alias/exact     | Similarity: 1.0
Input: py                   | Canonical: Python                    | Method: alias/exact     | Similarity: 1.0
Input: python3              | Canonical: Python                    | Method: alias/exact     | Similarity: 1.0
Input: k8s                  | Canonical: Kubernetes                | Method: alias/exact     | Similarity: 1.0
Input: kube                 | Canonical: Kubernetes                | Method: alias/exact     | Similarity: 1.0
Input: scikit-leran         | Canonical: Scikit-Learn              | Method: alias/exact     | Similarity: 1.0
